# Calculation of Wc  

In [2]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [13]:
dsU = xr.open_dataset(r'../data/UVW/U_2022.nc')


In [10]:
dsV = xr.open_dataset(r'../data/UVW/V_2022.nc').vo
dsV

<xarray.DataArray 'vo' (time: 1, depth: 50, latitude: 1201, longitude: 1201)> Size: 288MB
[72120050 values with dtype=float32]
Coordinates:
  * depth      (depth) float32 200B 0.494 1.541 2.646 ... 5.275e+03 5.728e+03
  * latitude   (latitude) float32 5kB -50.0 -49.92 -49.83 ... 49.83 49.92 50.0
  * longitude  (longitude) float32 5kB -100.0 -99.92 -99.83 ... -0.08333 0.0
  * time       (time) datetime64[ns] 8B 2022-06-01
Attributes:
    unit_long:      Meters per second
    standard_name:  northward_sea_water_velocity
    long_name:      Northward velocity
    valid_max:      5.0
    valid_min:      -5.0
    units:          m s-1

In [14]:
dsU

<xarray.Dataset> Size: 288MB
Dimensions:    (depth: 50, latitude: 1201, longitude: 1201, time: 1)
Coordinates:
  * depth      (depth) float32 200B 0.494 1.541 2.646 ... 5.275e+03 5.728e+03
  * latitude   (latitude) float32 5kB -50.0 -49.92 -49.83 ... 49.83 49.92 50.0
  * longitude  (longitude) float32 5kB -100.0 -99.92 -99.83 ... -0.08333 0.0
  * time       (time) datetime64[ns] 8B 2022-06-01
Data variables:
    uo         (time, depth, latitude, longitude) float32 288MB ...
Attributes:
    references:                http://marine.copernicus.eu
    source:                    MOI GLO12
    contact:                   https://marine.copernicus.eu/contact
    credit:                    E.U. Copernicus Marine Service Information (CM...
    producer:                  CMEMS - Global Monitoring and Forecasting Centre
    Conventions:               CF-1.8
    institution:               Mercator Ocean International
    title:                     daily mean fields from Global Ocean Physics An...
    copernicusmarine_version:  2.1.2

In [ ]:
## Mask calculation
_lat = ds.latitude
_lon = ds.longitude
_zt = ds.depth

ds = ds.rename({"depth": "k", "latitude":"j", "longitude":"i"})
ds = ds.assign_coords(
    k=np.arange(ds.sizes["k"]),
    j=np.arange(ds.sizes["j"]),
    i=np.arange(ds.sizes["i"]),
    depth_t=("k", _zt.data),
    latitude_f = ("j", _lat.data),
    longitude_f = ("i", _lon.data),
)

ds = ds.rename({"uo":"uf", "vo":"vf"})

## Calculate F and T mask
ds = ds.assign(fmask = ds.uf.isel(time=0,drop=True).notnull())

ds = ds.assign(
    tmask=(
        ds.fmask.shift(i=0,j=0)
        | ds.fmask.shift(i=-1,j=-1).fillna(False)
        | ds.fmask.shift(i=0, j=-1).fillna(False)
        | ds.fmask.shift(i=-1,j=1).fillna(False)
    ).astype(bool)
)

## Calculate U and V faces
ds = ds.assign(
    u=(ds.uf.fillna(0) + ds.uf.shift(j=-1).fillna(0)) /2,
    v=(ds.vf.fillna(0) + ds.vf.shift(i=-1).fillna(0)) /2,
)

## Calculate Zt
zt = ds.depth_t.data
zw = [zt[0]*2]


for k in range(1,50):
    zw.append((zt[k] - zw[k-1])*2 + zw[k-1])

ds = ds.assign_coords(depth_w = ("k",zw))

ds = ds.assign_coords(
    longitude_u = ds.longitude_f,
    latitude_v =  ds.latitude_f,
    
    latitude_u = ds.latitude_f + 1/12/2, 
    longitude_v = ds.longitude_f + 1/12/2,
    
    latitude_t = ds.latitude_f + 1/12/2, 
    longitude_t = ds.longitude_f + 1/12/2,
)

R = 6371e3 

ds = ds.assign_coords(
    dz_t = ds.depth_w - ds.depth_w.shift(k=1).fillna(0), 
    dx_t = np.deg2rad(1/12) * R * np.cos(np.deg2rad(ds.latitude_t)),
    dy_t = np.deg2rad(1/12) * R ,
    
)

## we find the total volume flux - m3
F_uv_vol = (
    ds.u * ds.dy_t * ds.dz_t - ds.u.shift(i=-1)* ds.dy_t * ds.dz_t 
    + ds.v * ds.dx_t * ds.dz_t - ds.v.shift(j=-1) * ds.dx_t * ds.dz_t
).fillna(0)

#we divide the total flux by the volume (dx*dy*dz) - 1/s
dw_by_dz = -F_uv_vol/ds.dx_t/ds.dy_t/ds.dz_t

w = (dw_by_dz.fillna(0) * ds.dz_t.fillna(0)).cumsum('k').fillna(0).where(ds.tmask==1)
w_bottom=w.isel(k=ds.tmask.sum('k')-1)
w_correct = w - w_bottom / ds.dz_t.where(ds.tmask==1).sum('k') * ds.depth_w

In [ ]:
w_correct.drop_encoding().to_netcdf('/work/bk1450/b383184/Amazon/Divergence/data/C_grid.nc')